# CAS Exam 5: Triangle Diagnostics & Data Adjustments
## (Non-Berquist-Sherman Adjustments)

**Scope:** This notebook covers the full diagnostic toolkit an actuary applies *before and during* method selection — everything beyond the two core Berquist-Sherman adjustments (settlement rate and case adequacy changes).

**Cross-reference:** For the B-S paid and reported adjustments, see `exam5_berquistshermanadjustments.ipynb`.

---

## Diagnostic Decision Matrix

| Trigger | Test | Finding | Action |
|---|---|---|---|
| Distorted diagonal | L/S test | pct_L ≥ 0.75 or ≤ 0.25 | Truncate n_periods or drop valuation |
| Volatile link ratios | CV analysis | CV > 15% | Investigate; use fewer periods or medial avg |
| Exposure/premium growth | Diagonal growth rate | > 10%/yr | Normalize triangle for exposure |
| Outlier claims | Incremental inspection | Cell > mean + 2σ | Cap; compare capped vs uncapped IBNR |
| Paid ≠ incurred development | Paid/incurred comparison | LDF ratio trending | B-S if operational; method selection if structural |
| Rising/falling case reserves | Case OS trend | Positive/negative slope | Prefer paid-basis or incurred-basis methods |

## Contents
1. Setup
2. Calendar Year (Diagonal) Analysis
3. Link Ratio Stability Analysis
4. Growth Analysis
5. Large Loss Analysis
6. Paid vs Incurred Comparison
7. Case Outstanding Trend
8. Sensitivity Testing
9. Method Comparison Framework

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import chainladder as cl

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import (
    run_chain_ladder,
    run_bornhuetter_ferguson,
    paid_vs_incurred_comparison,
    calendar_year_diagnostic,
    link_ratio_table,
    plot_link_ratio_heatmap,
    trend_summary,
    snapshot_method_output,
    build_reconciliation_report,
)

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
pd.set_option('display.max_columns', 15)

# Load sample — quarterly grain, 12 accident years (1995–2006), paid + incurred
_raw     = cl.load_sample('quarterly')
paid     = _raw['paid']
incurred = _raw['incurred']

print(f'Paid triangle shape:     {paid.shape}')
print(f'Incurred triangle shape: {incurred.shape}')
print(f'Valuation date:          {paid.valuation_date.strftime("%Y-%m-%d")}')
print(f'Development grain:       quarterly (3-month steps)')
print(f'Development range:       {int(paid.development[0])} – {int(paid.development[-1])} months')

---
## Section 1 — Calendar Year (Diagonal) Analysis

### Theory

**Calendar year effects** are external forces that distort development uniformly across all accident years in the same diagonal. Examples: inflation shocks, legislative changes, claim-handling practice shifts, court decisions.

When present, simple volume-weighted averages will blend distorted and undistorted diagonals, producing biased LDFs.

### The L/S Test (Friedland Chapter 8)

For each development transition column:
1. Compute the column median link ratio.
2. Label each AY's ratio **L** (above median) or **S** (at or below median).
3. Map each L/S to its **calendar year** = origin year + later development age.
4. Sum L's and S's by calendar year.
5. Flag any calendar year with **pct_L ≥ 0.75** (above-average, possible superimposed inflation) or **pct_L ≤ 0.25** (below-average, reserve strengthening or slow payment).

### Decision Rule
- Flagged recent diagonal → truncate `n_periods` to exclude it, or use `drop_valuation`.
- Flagged older diagonal → consider whether it's a one-time event; may be safe to include.

In [ ]:
# Calendar year diagnostic — paid triangle
cy_paid = calendar_year_diagnostic(paid)

print('=== L/S MATRIX (paid) — first 10 development transitions ===')
print(cy_paid['ls_matrix'].iloc[:, :10].to_string())

print('\n=== CALENDAR YEAR SUMMARY (paid) ===')
print(cy_paid['calendar_year_summary'].to_string())
print('\n  pct_L ≥ 0.75 → above-average development (possible superimposed inflation)')
print('  pct_L ≤ 0.25 → below-average development (reserve strengthening or slow payment)')

In [ ]:
# Calendar year diagnostic — incurred triangle
cy_incurred = calendar_year_diagnostic(incurred)

print('=== L/S MATRIX (incurred) — first 10 development transitions ===')
print(cy_incurred['ls_matrix'].iloc[:, :10].to_string())

print('\n=== CALENDAR YEAR SUMMARY (incurred) ===')
print(cy_incurred['calendar_year_summary'].to_string())

In [ ]:
# Visualize pct_L by calendar year for paid
cy_sum = cy_paid['calendar_year_summary'].copy()
cy_sum = cy_sum[cy_sum['n_total'] >= 2]  # exclude thin years

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(cy_sum.index.astype(str), cy_sum['pct_L'],
               color=['#d62728' if p >= 0.75 else '#1f77b4' if p <= 0.25 else '#aec7e8'
                      for p in cy_sum['pct_L']], alpha=0.85, edgecolor='white')
ax.axhline(0.75, color='red', linestyle='--', linewidth=1, label='75% threshold (inflation signal)')
ax.axhline(0.25, color='blue', linestyle='--', linewidth=1, label='25% threshold (slow payment)')
ax.axhline(0.50, color='gray', linestyle=':', linewidth=0.8, label='Neutral (50%)')
ax.set_xlabel('Calendar Year')
ax.set_ylabel('pct_L (fraction of above-median link ratios)')
ax.set_title('Calendar Year L/S Diagnostic — Paid Triangle')
ax.legend(fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Practice Problem 1: L/S Matrix by Hand

Given the following 5×5 cumulative paid claims triangle:

| AY \ Age | 12 | 24 | 36 | 48 | 60 |
|---|---|---|---|---|---|
| 2019 | 1,000 | 1,600 | 2,100 | 2,450 | 2,650 |
| 2020 | 1,050 | 1,680 | 2,200 | 2,560 | — |
| 2021 | 1,080 | 1,800 | 2,350 | — | — |
| 2022 | 1,150 | 1,720 | — | — | — |
| 2023 | 1,200 | — | — | — | — |

**(a)** Compute the 12-24, 24-36, and 36-48 link ratios for each available AY.

**(b)** Classify each link ratio as L (above column median) or S (at/below column median).

**(c)** Map each L/S label to its calendar year (= AY + later dev age in years − 1). Count L's and S's by calendar year.

**(d)** Is there evidence of a calendar year effect? Which calendar year is most suspicious, and what action would you take?

In [ ]:
# Practice Problem 1 — Solution
import numpy as np
import pandas as pd

cum = pd.DataFrame({
    12: [1000, 1050, 1080, 1150, 1200],
    24: [1600, 1680, 1800, 1720, np.nan],
    36: [2100, 2200, 2350, np.nan, np.nan],
    48: [2450, 2560, np.nan, np.nan, np.nan],
    60: [2650, np.nan, np.nan, np.nan, np.nan],
}, index=[2019, 2020, 2021, 2022, 2023])

# (a) Link ratios
transitions = [(12, 24), (24, 36), (36, 48)]
lf = pd.DataFrame(index=cum.index)
for a1, a2 in transitions:
    lf[f'{a1}-{a2}'] = cum[a2] / cum[a1]

print('=== (a) Link Ratios ===')
print(lf.to_string(float_format=lambda x: f'{x:.4f}'))

# (b) L/S classification
ls = pd.DataFrame(index=cum.index)
for col in lf.columns:
    med = lf[col].median()
    ls[col] = lf[col].apply(lambda x: 'L' if pd.notna(x) and x > med else ('S' if pd.notna(x) else ''))

print('\n=== (b) L/S Classification (median per column) ===')
medians = {col: lf[col].median() for col in lf.columns}
print(f'  Medians: {medians}')
print(ls.to_string())

# (c) Calendar year mapping: CY = AY + later_age_years - 1
age_map = {(12, 24): 24, (24, 36): 36, (36, 48): 48}
records = []
for (a1, a2) in transitions:
    col = f'{a1}-{a2}'
    for ay in cum.index:
        label = ls.at[ay, col]
        if label in ('L', 'S'):
            cy = ay + a2 // 12 - 1
            records.append({'CY': cy, 'label': label})

cy_df = pd.DataFrame(records)
cy_summary = cy_df.groupby('CY')['label'].value_counts().unstack(fill_value=0)
cy_summary.columns.name = None
cy_summary['n_total'] = cy_summary.get('L', 0) + cy_summary.get('S', 0)
cy_summary['pct_L'] = cy_summary.get('L', 0) / cy_summary['n_total']
print('\n=== (c) Calendar Year Summary ===')
print(cy_summary.to_string(float_format=lambda x: f'{x:.2f}'))

# (d) Interpretation
flagged = cy_summary[cy_summary['pct_L'] >= 0.75]
print('\n=== (d) Flagged Calendar Years (pct_L ≥ 0.75) ===')
if len(flagged):
    for cy, row in flagged.iterrows():
        print(f'  CY {cy}: pct_L = {row["pct_L"]:.2f} — above-average development.')
    print('  Action: truncate n_periods to exclude this diagonal, '
          'or use drop_valuation to remove the flagged calendar year.')
else:
    print('  No calendar years exceed the 0.75 threshold — no action required.')

---
## Section 2 — Link Ratio Stability Analysis

### Theory

**Link ratio stability** measures how consistent age-to-age factors are across accident years for a given development transition. Instability can indicate:
- Random noise (small triangle, sparse data)
- Structural changes in claim settlement or reporting patterns
- Large losses in specific accident years

### Coefficient of Variation (CV)

$$CV = \frac{\sigma}{\mu}$$

where σ and μ are the standard deviation and mean of link ratios for a given transition across accident years.

| CV Range | Stability Assessment | Action |
|---|---|---|
| CV < 5% | Highly stable | Use all-period average |
| CV 5–15% | Moderate variability | Consider n_periods restriction or medial avg |
| CV > 15% | High variability | Investigate causes; restrict or exclude outliers |

### Three Averaging Options
- **Volume-weighted (vol-wtd):** Σ numerators / Σ denominators — appropriate when large AYs should dominate.
- **Simple average:** arithmetic mean — gives equal weight to each AY regardless of size.
- **Medial average:** drop the single highest and lowest, then simple-average the rest — robust to outliers.

In [ ]:
# Full link ratio exhibit — paid triangle (first 8 transitions shown)
ldf_table = link_ratio_table(paid, n_periods=-1)

print('=== AGE-TO-AGE FACTOR EXHIBIT (paid) — first 8 transitions ===')
print(ldf_table.iloc[:, :8].to_string(float_format=lambda x: f'{x:.4f}' if pd.notna(x) else ''))

In [ ]:
# Link ratio heatmap (first 10 transitions for readability)
# We build a trimmed triangle covering first 10 development periods
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_link_ratio_heatmap(paid,     ax=axes[0], title='Paid — Link Ratio Heatmap')
plot_link_ratio_heatmap(incurred, ax=axes[1], title='Incurred — Link Ratio Heatmap')
plt.suptitle('Green = fast development (below median)  |  Red = slow development (above median)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Compute CV for each development transition (paid triangle)
# Only factor rows (exclude summary rows)
summary_labels = {'Vol Wtd Avg', 'Simple Avg', 'Medial Avg', 'Selected LDF'}
factor_only = ldf_table.loc[~ldf_table.index.isin(summary_labels)].astype(float)

cv_stats = pd.DataFrame({
    'mean':   factor_only.mean(),
    'std':    factor_only.std(),
    'cv_pct': (factor_only.std() / factor_only.mean() * 100).round(1),
    'n_obs':  factor_only.notna().sum(),
})

def _stability(cv):
    if pd.isna(cv):    return 'n/a'
    if cv < 5:         return 'Stable — use all periods'
    if cv < 15:        return 'Moderate — consider n_periods restriction'
    return 'High — investigate; medial or restricted avg'

cv_stats['stability'] = cv_stats['cv_pct'].apply(_stability)

print('=== COEFFICIENT OF VARIATION BY TRANSITION (paid, first 10) ===')
print(cv_stats.iloc[:10].to_string(float_format=lambda x: f'{x:.4f}'))

# n_periods comparison (all vs 5 vs 3)
print('\n=== AVERAGE LDF SENSITIVITY TO n_periods (paid, first 8 transitions) ===')
rows = {}
for n, label in [(-1, 'All periods'), (5, 'Last 5'), (3, 'Last 3')]:
    t = link_ratio_table(paid, n_periods=n)
    rows[label] = t.loc['Selected LDF'].iloc[:8]
pd.DataFrame(rows).T.style.format('{:.4f}')
comparison_df = pd.DataFrame(rows).T
print(comparison_df.to_string(float_format=lambda x: f'{x:.4f}'))

### Practice Problem 2: Link Ratio Outlier Detection

The following are observed 24-36 link ratios for five accident years:

| AY | Paid at 24 mo | Paid at 36 mo | LDF 24-36 |
|---|---|---|---|
| 2015 | 4,200 | 5,513 | ? |
| 2016 | 4,450 | 5,787 | ? |
| 2017 | 4,100 | 6,753 | ? ← suspected large loss |
| 2018 | 4,600 | 5,981 | ? |
| 2019 | 4,800 | 6,192 | ? |

**(a)** Compute all five LDFs.

**(b)** Apply the mean + 2σ outlier rule. Is AY 2017 an outlier?

**(c)** Compute (i) the vol-wtd average using all 5 AYs and (ii) excluding AY 2017.

**(d)** What is the percent change in the selected 24-36 LDF, and how would this propagate to IBNR?

In [ ]:
# Practice Problem 2 — Solution
pp2 = pd.DataFrame({
    'paid_24': [4200, 4450, 4100, 4600, 4800],
    'paid_36': [5513, 5787, 6753, 5981, 6192],
}, index=[2015, 2016, 2017, 2018, 2019])

# (a) LDFs
pp2['ldf_24_36'] = pp2['paid_36'] / pp2['paid_24']
print('=== (a) LDFs ===')
print(pp2.to_string(float_format=lambda x: f'{x:.4f}'))

# (b) Outlier detection
mu  = pp2['ldf_24_36'].mean()
sig = pp2['ldf_24_36'].std()
cap = mu + 2 * sig
print(f'\n=== (b) Outlier Rule: mean + 2σ ===')
print(f'  Mean LDF:  {mu:.4f}')
print(f'  Std Dev:   {sig:.4f}')
print(f'  Cap (μ+2σ): {cap:.4f}')
for ay, row in pp2.iterrows():
    flag = ' ← OUTLIER' if row['ldf_24_36'] > cap else ''
    print(f'  AY {ay}: {row["ldf_24_36"]:.4f}{flag}')

# (c) Vol-wtd averages
vw_all    = pp2['paid_36'].sum() / pp2['paid_24'].sum()
pp2_trim  = pp2.drop(2017)
vw_no2017 = pp2_trim['paid_36'].sum() / pp2_trim['paid_24'].sum()
print(f'\n=== (c) Vol-weighted Averages ===')
print(f'  All 5 AYs:        {vw_all:.4f}')
print(f'  Excl. AY 2017:    {vw_no2017:.4f}')

# (d) Impact
pct_chg = (vw_no2017 / vw_all - 1) * 100
print(f'\n=== (d) Impact of Removing AY 2017 ===')
print(f'  LDF change: {pct_chg:+.2f}% — a lower 24-36 LDF means lower IBNR for immature AYs.')
print(f'  For an AY with $5,000 at 24 months:')
print(f'    Projected to 36 mo (with outlier):    {5000 * vw_all:,.0f}')
print(f'    Projected to 36 mo (without outlier): {5000 * vw_no2017:,.0f}')
print(f'    Difference (IBNR impact at this age): {5000 * (vw_all - vw_no2017):,.0f}')

---
## Section 3 — Growth Analysis

### Theory

**Premium or exposure growth** can make a triangle appear to develop faster than it actually will. When earned exposure grows year over year, the latest diagonal (most recent accident year at each age) contains more exposure than older accident years did at the same age. This means:

- Newer AYs produce more absolute losses at the same maturity
- Volume-weighted LDFs are pulled toward the patterns of the newest (larger) AYs
- Development patterns appear faster than the underlying claim settlement process

### Detection: Diagonal Trend Analysis

Plot the **latest diagonal** (each AY's most recent observed amount) over accident years. If this plot shows systematic growth beyond claim severity trend, the triangle is subject to exposure growth bias.

### Trend Summary Diagnostics

- **age_trends:** slope of link ratios across development ages for each origin year — detects whether factors are declining as each AY matures (normal) or increasing (unusual).
- **origin_trends:** slope of link ratios across origin years for each development age — detects whether factors are trending up or down over accident years (trend in patterns).
- **diagonal_trend:** slope of average link ratio by calendar year — detects calendar year systematic effects.

In [ ]:
# Trend summary diagnostics
try:
    ts_paid = trend_summary(paid)
    print('=== AGE TRENDS (slope of link ratios across dev ages per AY) ===')
    print('  Negative = factors declining across ages (normal LDF convergence)')
    print(ts_paid['age_trends'].to_frame('slope').T.to_string(float_format=lambda x: f'{x:.6f}'))

    print('\n=== ORIGIN TRENDS (slope of link ratios across AYs per dev age, first 8) ===')
    print('  Negative = factors declining over accident years (favors recent weighting)')
    print(ts_paid['origin_trends'].iloc[:8].to_frame('slope').T.to_string(float_format=lambda x: f'{x:.6f}'))

    print('\n=== DIAGONAL TREND (slope of avg link ratio by calendar year) ===')
    print(ts_paid['diagonal_trend'].to_string(float_format=lambda x: f'{x:.6f}'))

    # Interpretation
    diag_slope = ts_paid['diagonal_trend']['slope'].iloc[0]
    if abs(diag_slope) > 0.002:
        direction = 'upward' if diag_slope > 0 else 'downward'
        print(f'\n  *** Diagonal slope = {diag_slope:.4f} ({direction} trend) — '
              f'possible calendar year effect; review L/S matrix in Section 1.')
    else:
        print(f'\n  Diagonal slope = {diag_slope:.4f} — no strong systematic calendar year trend.')

except Exception as e:
    print(f'trend_summary: {e}')

In [ ]:
# Plot latest diagonal over accident years — detect exposure growth
paid_wide = paid.to_frame(origin_as_datetime=False)
latest_diag = paid_wide.apply(lambda row: row.dropna().iloc[-1] if row.notna().any() else np.nan, axis=1)
latest_age  = paid_wide.apply(lambda row: row.dropna().index[-1] if row.notna().any() else np.nan, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(range(len(latest_diag)), latest_diag.values, marker='o', color='steelblue')
ax.set_xticks(range(len(latest_diag)))
ax.set_xticklabels(latest_diag.index.astype(str), rotation=45, fontsize=8)
ax.set_title('Latest Diagonal Paid by Accident Year')
ax.set_ylabel('Paid Claims')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, alpha=0.3)

# Year-over-year growth rate
yoy = latest_diag.pct_change() * 100
ax2 = axes[1]
colors = ['#d62728' if v > 10 else '#1f77b4' for v in yoy.values[1:]]
ax2.bar(range(1, len(yoy)), yoy.values[1:], color=colors, alpha=0.8, edgecolor='white')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axhline(10, color='red', linestyle='--', linewidth=1, label='>10% growth (exposure bias risk)')
ax2.set_xticks(range(1, len(yoy)))
ax2.set_xticklabels(latest_diag.index[1:].astype(str), rotation=45, fontsize=8)
ax2.set_title('YoY Growth Rate in Latest Diagonal')
ax2.set_ylabel('Growth % (year over year)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle('Exposure Growth Diagnostic: Latest Diagonal Analysis', y=1.02)
plt.tight_layout()
plt.show()

print('Year-over-year growth in latest diagonal paid:')
print(yoy.to_frame('YoY Growth %').to_string(float_format=lambda x: f'{x:.1f}'))

### Practice Problem 3: Growth-Adjusted Development

A paid triangle shows the following latest-diagonal cumulative paid claims at age **36 months**:

| AY | Paid at 36 mo | Exposure Index (2019=1.00) |
|---|---|---|
| 2019 | 2,000,000 | 1.000 |
| 2020 | 2,300,000 | 1.150 |
| 2021 | 2,645,000 | 1.323 |
| 2022 | 3,042,000 | 1.521 |
| 2023 | 3,498,000 | 1.749 |

Assume the underlying book has grown 15% per year in exposure, all else equal.

**(a)** Index each AY's paid-at-36 to a 2019 cost/exposure level by dividing by the exposure index.

**(b)** The 24-36 link ratios (unadjusted) are 1.320, 1.340, 1.350, 1.360, 1.375. What bias does rapid growth introduce into these unadjusted LDFs?

**(c)** After indexing, compute adjusted LDFs if the paid at 24 months (adjusted) are 1,515, 1,515, 1,515, 1,515, 1,515 for all AYs.

In [ ]:
# Practice Problem 3 — Solution
pp3 = pd.DataFrame({
    'paid_36_raw': [2_000_000, 2_300_000, 2_645_000, 3_042_000, 3_498_000],
    'exp_index':   [1.000, 1.150, 1.150**2, 1.150**3, 1.150**4],
    'ldf_24_36_unadj': [1.320, 1.340, 1.350, 1.360, 1.375],
    'paid_24_adj': [1_515_000] * 5,
}, index=[2019, 2020, 2021, 2022, 2023])

# (a) Index to 2019 level
pp3['paid_36_adj'] = (pp3['paid_36_raw'] / pp3['exp_index']).round(0)
print('=== (a) Paid-at-36 Indexed to 2019 Exposure Level ===')
print(pp3[['paid_36_raw', 'exp_index', 'paid_36_adj']].to_string(float_format=lambda x: f'{x:,.0f}'))

# (b) Bias explanation
print('\n=== (b) Bias from Rapid Growth ===')
print('  Volume-weighted LDF uses Σ(paid_36) / Σ(paid_24).')
print('  With 15% growth, each successive AY has 15% more exposure.')
print('  Newer AYs dominate the sum — if newer AYs have HIGHER factors (growth bias),')
print('  the vol-wtd average is pulled UP, overstating expected future development.')
print(f'  Unadjusted LDF trend: {pp3["ldf_24_36_unadj"].tolist()} — clearly rising across AYs.')

# (c) Adjusted LDFs
pp3['ldf_24_36_adj'] = pp3['paid_36_adj'] / pp3['paid_24_adj']
vw_adj = pp3['paid_36_adj'].sum() / pp3['paid_24_adj'].sum()
vw_unadj_approx = 1.349  # rough average of unadjusted

print('\n=== (c) Adjusted LDFs ===')
print(pp3[['paid_24_adj', 'paid_36_adj', 'ldf_24_36_adj']].to_string(float_format=lambda x: f'{x:,.4f}'))
print(f'\n  Vol-wtd adjusted LDF (24-36): {vw_adj:.4f}')
print(f'  Unadjusted avg was ~{pp3["ldf_24_36_unadj"].mean():.4f}')
print(f'  Growth adjustment removes ~{(pp3["ldf_24_36_unadj"].mean() - vw_adj):.4f} from the LDF.')

---
## Section 4 — Large Loss Analysis

### Theory

**Large individual losses** distort incremental paid development. A single catastrophic payment in one accident year's incremental triangle can inflate the development factor for that period, causing an upward bias in projected IBNR.

### Process
1. Convert cumulative triangle to **incremental** using `cum_to_incr()`.
2. Identify outlier cells: values > column mean + 2σ (or some threshold).
3. **Cap** or remove the anomalous values; document the removed amounts.
4. Reconstruct the cumulative triangle from the capped incremental.
5. Run chain ladder on both capped and uncapped; compare IBNR estimates.
6. The difference is the **large loss IBNR adjustment**.

### Credibility Blending (advanced)
Rather than a hard cap, blend the scrubbed and unscrubbed patterns:
$$LDF_{selected} = z \cdot LDF_{capped} + (1-z) \cdot LDF_{uncapped}$$
where z is the credibility weight assigned to the capped pattern.

In [ ]:
# Large loss analysis on the paid triangle
incr_paid = paid.cum_to_incr()

# Wide format: origins × development ages
incr_wide = incr_paid.to_frame(origin_as_datetime=False)

print('=== INCREMENTAL PAID TRIANGLE (first 8 dev columns) ===')
print(incr_wide.iloc[:, :8].to_string(float_format=lambda x: f'{x:,.0f}' if pd.notna(x) else ''))

# Identify outliers: cells > column mean + 2*std
outlier_flags = pd.DataFrame(False, index=incr_wide.index, columns=incr_wide.columns)
caps = {}
for col in incr_wide.columns:
    col_data = incr_wide[col].dropna()
    if len(col_data) < 3:
        continue
    mu_c  = col_data.mean()
    sig_c = col_data.std()
    cap_c = mu_c + 2 * sig_c
    caps[col] = cap_c
    outlier_flags[col] = incr_wide[col] > cap_c

n_outliers = outlier_flags.sum().sum()
print(f'\nOutlier cells (value > col mean + 2σ): {n_outliers}')
if n_outliers > 0:
    for col in incr_wide.columns[:8]:
        flagged = outlier_flags[col][outlier_flags[col]].index.tolist()
        if flagged:
            cap_v = caps.get(col, np.nan)
            for ay in flagged:
                val = incr_wide.at[ay, col]
                print(f'  AY {ay}, dev col {col}: incremental = {val:,.0f}, cap = {cap_v:,.0f}')

In [ ]:
# Compare CL IBNR: n_periods=3 (recent, less impacted by any one large year)
# vs n_periods=1 (only most recent — most impacted by recent large losses)
ibnr_by_n = {}
for n, label in [(-1, 'All periods'), (5, 'Last 5'), (3, 'Last 3'), (1, 'Last 1')]:
    try:
        r, s, _ = run_chain_ladder(paid, development_kwargs={'n_periods': n})
        ibnr_by_n[label] = s['ibnr']
    except Exception as e:
        print(f'  n={n} failed: {e}')

if ibnr_by_n:
    ibnr_df = pd.DataFrame(ibnr_by_n)
    ibnr_df.loc['Total'] = ibnr_df.sum()
    print('=== IBNR BY ACCIDENT YEAR vs n_periods (proxy for large loss sensitivity) ===')
    print(ibnr_df.to_string(float_format=lambda x: f'{x:,.0f}'))

    print('\n=== TOTAL IBNR ===')
    for label, s_df in ibnr_by_n.items():
        print(f'  {label:<15}: {s_df.sum():>12,.0f}')
    print('\nLarge spread between n=1 and n=All suggests at least one large-loss year'
          ' heavily influences the short-period average.')

### Practice Problem 4: Large Loss Cap and Re-projection

The following incremental paid triangle (in $000s) has a suspected large loss in AY 2021 at the 24-36 development transition:

| AY \ Dev | 12 | 12→24 | 24→36 | 36→48 |
|---|---|---|---|---|
| 2019 | 800 | 620 | 300 | 150 |
| 2020 | 850 | 640 | 310 | — |
| 2021 | 870 | 650 | **2,800** | — |
| 2022 | 900 | 680 | — | — |
| 2023 | 920 | — | — | — |

**(a)** For the 24-36 incremental column (excluding AY 2021), compute the mean and standard deviation.

**(b)** Apply the mean + 2σ cap. What is the capped value for AY 2021?

**(c)** Construct the capped cumulative triangle and compute vol-wtd 24-36 and 36-48 LDFs.

**(d)** Project AY 2021 and AY 2022 to ultimate under the capped pattern. What is the IBNR difference from using uncapped factors?

In [ ]:
# Practice Problem 4 — Solution
incr = pd.DataFrame({
    '12':    [800, 850, 870, 900, 920],
    '12-24': [620, 640, 650, 680, np.nan],
    '24-36': [300, 310, 2800, np.nan, np.nan],
    '36-48': [150, np.nan, np.nan, np.nan, np.nan],
}, index=[2019, 2020, 2021, 2022, 2023])

# (a) 24-36 column stats (excluding AY 2021)
col_24_36 = incr['24-36'].drop(2021).dropna()
mu_ll  = col_24_36.mean()
sig_ll = col_24_36.std()
cap_ll = mu_ll + 2 * sig_ll
print(f'=== (a) 24-36 incremental column (excl. AY 2021) ===')
print(f'  Values: {col_24_36.tolist()}')
print(f'  Mean:   {mu_ll:.1f}')
print(f'  Std:    {sig_ll:.1f}')
print(f'  Cap (μ+2σ): {cap_ll:.1f}')

# (b) Capped value
raw_2021 = incr.at[2021, '24-36']
capped_2021 = min(raw_2021, cap_ll)
print(f'\n=== (b) AY 2021 cap ===')
print(f'  Raw incremental:   {raw_2021:,.0f}')
print(f'  Capped incremental: {capped_2021:.1f}')
print(f'  Amount removed:    {raw_2021 - capped_2021:,.1f}')

# (c) Capped cumulative triangle and LDFs
incr_capped = incr.copy()
incr_capped.at[2021, '24-36'] = capped_2021

# Reconstruct cumulative
cum_capped = pd.DataFrame(index=incr.index)
cum_capped[12] = incr_capped['12']
cum_capped[24] = cum_capped[12] + incr_capped['12-24']
cum_capped[36] = cum_capped[24] + incr_capped['24-36']
cum_capped[48] = cum_capped[36] + incr_capped['36-48']

# Vol-wtd LDFs from capped
ldf_24_36_cap = cum_capped[36].dropna().sum() / cum_capped[24].dropna().sum()
ldf_36_48_cap = cum_capped[48].dropna().sum() / cum_capped[36].dropna().sum()
print(f'\n=== (c) Capped Cumulative Triangle ===')
print(cum_capped.to_string(float_format=lambda x: f'{x:,.1f}'))
print(f'  Capped vol-wtd LDF 24-36: {ldf_24_36_cap:.4f}')
print(f'  Capped vol-wtd LDF 36-48: {ldf_36_48_cap:.4f}')

# Uncapped for comparison
cum_raw = pd.DataFrame(index=incr.index)
cum_raw[12] = incr['12']
cum_raw[24] = cum_raw[12] + incr['12-24']
cum_raw[36] = cum_raw[24] + incr['24-36']
cum_raw[48] = cum_raw[36] + incr['36-48']
ldf_24_36_raw = cum_raw[36].dropna().sum() / cum_raw[24].dropna().sum()
ldf_36_48_raw = cum_raw[48].dropna().sum() / cum_raw[36].dropna().sum()

# (d) IBNR comparison for AY 2021 and 2022
print(f'\n=== (d) IBNR Comparison — Capped vs Uncapped ===')
for ay in [2021, 2022]:
    latest_c = cum_capped.loc[ay].dropna().iloc[-1]
    latest_r = cum_raw.loc[ay].dropna().iloc[-1]
    latest_age = cum_capped.loc[ay].dropna().index[-1]
    cdf_c = ldf_24_36_cap * ldf_36_48_cap if latest_age == 24 else (ldf_36_48_cap if latest_age == 36 else 1.0)
    cdf_r = ldf_24_36_raw * ldf_36_48_raw if latest_age == 24 else (ldf_36_48_raw if latest_age == 36 else 1.0)
    ult_c = latest_c * cdf_c
    ult_r = latest_r * cdf_r
    print(f'  AY {ay} (latest age {latest_age} mo):')
    print(f'    Capped IBNR:   {ult_c - latest_c:,.1f}   (ult = {ult_c:,.1f})')
    print(f'    Uncapped IBNR: {ult_r - latest_r:,.1f}   (ult = {ult_r:,.1f})')
    print(f'    Difference:    {(ult_c - latest_c) - (ult_r - latest_r):+,.1f}')

---
## Section 5 — Paid vs Incurred Comparison

### Theory

Comparing paid and incurred development patterns is a key pre-selection diagnostic. The key metrics are:

| Pattern | Interpretation | Implication |
|---|---|---|
| Paid LDF > Incurred LDF | Paid settling faster than incurred adds reserves | Settlement rate speedup (B-S paid signal) |
| Paid LDF < Incurred LDF | Normal — incurred always leads paid | Case reserves developing as expected |
| LDF ratio (paid/incurred) trending UP | Settlement accelerating over time | B-S paid adjustment may be needed |
| LDF ratio trending DOWN | Incurred developing faster than paid | Case reserve strengthening (B-S incurred signal) |
| Paid/incurred ratio at latest diagonal declining | Case adequacy rising | Incurred is being restated upward relative to paid |

### Beyond B-S
Even when B-S adjustments are NOT warranted, the paid/incurred comparison informs **method selection**:
- If paid and incurred produce materially different ultimates, investigate why before selecting a method.
- Rising case OS levels → lean toward paid-basis projections (incurred may overstate future development).
- Munich Chain Ladder is an alternative that explicitly constrains paid and incurred to converge toward the same ultimate.

In [ ]:
# Paid vs incurred comparison
pvi = paid_vs_incurred_comparison(paid, incurred, n_periods=-1)

print('=== LDF COMPARISON (first 10 development ages) ===')
print('  ldf_ratio = paid_ldf / incurred_ldf')
print('  < 1.0 = paid develops slower (incurred leads paid — normal at early ages)')
print(pvi['ldf_comparison'].iloc[:10].to_string(float_format=lambda x: f'{x:.4f}'))

print('\n=== CDF COMPARISON (first 10 development ages) ===')
print('  cdf_ratio = paid_cdf / incurred_cdf')
print('  Should converge toward 1.0 as CDFs both approach 1.0 at later ages')
print(pvi['cdf_comparison'].iloc[:10].to_string(float_format=lambda x: f'{x:.4f}'))

print('\n=== PAID/INCURRED AT LATEST DIAGONAL BY ORIGIN YEAR ===')
print(pvi['paid_to_incurred_latest'].to_string(float_format=lambda x: f'{x:,.2f}'))

In [ ]:
# Plot CDF comparison and paid/incurred ratio trend
cdf_comp = pvi['cdf_comparison'].dropna(how='all')
pi_latest = pvi['paid_to_incurred_latest']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: CDF comparison
ax = axes[0]
ax.plot(range(len(cdf_comp.iloc[:12])), cdf_comp['paid_cdf'].iloc[:12].values,
        marker='o', label='Paid CDF', color='steelblue')
ax.plot(range(len(cdf_comp.iloc[:12])), cdf_comp['incurred_cdf'].iloc[:12].values,
        marker='s', label='Incurred CDF', color='darkorange')
ax.set_xticks(range(len(cdf_comp.iloc[:12])))
ax.set_xticklabels(cdf_comp.index[:12].astype(str), rotation=45, fontsize=7)
ax.set_title('Paid vs Incurred CDF to Ultimate (first 12 ages)')
ax.set_ylabel('CDF to Ultimate')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: paid/incurred ratio at latest diagonal
ax2 = axes[1]
ay_labels = [str(x)[:4] for x in pi_latest.index]
ax2.bar(range(len(pi_latest)), pi_latest['paid_to_incurred'].values,
        color=['#d62728' if r < 0.8 else '#1f77b4' for r in pi_latest['paid_to_incurred'].values],
        alpha=0.85)
ax2.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Fully paid (ratio = 1.0)')
ax2.set_xticks(range(len(pi_latest)))
ax2.set_xticklabels(ay_labels, rotation=45, fontsize=8)
ax2.set_title('Paid / Incurred at Latest Diagonal by AY')
ax2.set_ylabel('Paid / Incurred Ratio')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Trend in paid/incurred ratio
pi_values = pi_latest['paid_to_incurred'].dropna().values
x_vals    = np.arange(len(pi_values))
slope_pi  = float(np.polyfit(x_vals, pi_values, 1)[0]) if len(pi_values) >= 2 else np.nan
print(f'Trend slope of paid/incurred ratio: {slope_pi:+.4f} per accident year')
if slope_pi < -0.005:
    print('  Declining ratio → case reserves strengthening over time.')
    print('  Investigate whether B-S reported adjustment is appropriate.')
elif slope_pi > 0.005:
    print('  Rising ratio → paid catching up to incurred faster over time.')
    print('  May indicate settlement rate acceleration — consider B-S paid adjustment.')
else:
    print('  Stable ratio → no strong structural shift detected.')

### Practice Problem 5: B-S Appropriateness Decision

The paid/incurred ratios at the **latest diagonal** for accident years 2018–2023 are:

| AY | Paid/Incurred |
|---|---|
| 2018 | 0.910 |
| 2019 | 0.880 |
| 2020 | 0.840 |
| 2021 | 0.820 |
| 2022 | 0.780 |
| 2023 | 0.750 |

**(a)** Compute the OLS trend slope of the paid/incurred ratio over accident years.

**(b)** The paid LDF ratios (paid_ldf / incurred_ldf) are all between 0.90 and 0.98 across development periods — no value exceeds 1.0. Does this suggest settlement rate acceleration?

**(c)** Is a B-S reported (incurred) adjustment warranted based on (a) and (b)? State your conclusion clearly.

**(d)** If B-S is not applied, what method selection strategy accounts for this pattern?

In [ ]:
# Practice Problem 5 — Solution
pp5 = pd.Series([0.910, 0.880, 0.840, 0.820, 0.780, 0.750],
                index=[2018, 2019, 2020, 2021, 2022, 2023], name='paid_to_incurred')

# (a) OLS trend slope
x = np.arange(len(pp5))
slope, intercept = np.polyfit(x, pp5.values, 1)
fitted = slope * x + intercept

print('=== (a) Trend Slope ===')
print(pp5.to_frame().to_string(float_format=lambda x: f'{x:.3f}'))
print(f'  OLS slope = {slope:+.4f} per accident year')
print(f'  Cumulative decline over 6 years: {slope * 5:.3f}')

# (b) LDF ratio analysis
print('\n=== (b) LDF Ratio Assessment ===')
print('  All paid_ldf / incurred_ldf ratios are 0.90–0.98 < 1.0.')
print('  This means paid is developing SLOWER than incurred at every age — normal pattern.')
print('  Settlement rate acceleration would show paid_ldf/incurred_ldf > 1.0 at early ages.')
print('  CONCLUSION: No evidence of settlement rate acceleration.')

# (c) B-S decision
print('\n=== (c) B-S Reported Adjustment — Decision ===')
print(f'  Declining paid/incurred ratio (slope = {slope:.4f}) suggests case reserves')
print('  are growing faster than paid claims across accident years.')
print('  This IS the classic signal for B-S reported (case adequacy) adjustment.')
print('  CONCLUSION: B-S reported adjustment IS warranted.')
print('  Method: restate historical incurred using current (latest) average case OS')
print('          trended backward at the selected severity trend rate.')

# (d) Method selection without B-S
print('\n=== (d) Alternative Without B-S ===')
print('  If B-S is not applied:')
print('  1. Use PAID-basis projections (chain ladder on paid) as the primary method.')
print('     Paid triangle is unaffected by case reserve strengthening.')
print('  2. Treat incurred-basis projections as a secondary/upper bound.')
print('  3. For immature AYs, use BF/Benktander on PAID claims.')
print('  4. Document the case adequacy trend as a qualitative caveat to the reserve.')  

---
## Section 6 — Case Outstanding Trend

### Theory

**Case outstanding (case OS)** = Reported claims − Paid claims at a given point in time. Tracking the *level and trend* of case OS separately from the B-S adjustment reveals how much of the incurred development is driven by case reserve changes vs. true claim settlement.

Key analyses:
- **Average case OS by development age**: if no count data, track dollar levels. Rising levels over accident years at the same age → strengthening.
- **Diagonal trend in case OS**: a rising trend across the latest diagonal (most recent AY at each age) suggests reserves are being systematically increased.
- **Comparison to paid development**: if case OS is rising but paid is flat → incurred will overstate future development → lean toward paid-basis methods.

### Method Selection Implication
- **Rising case OS trend** → incurred triangle is drifting upward → prefer paid-basis CL or BF.
- **Falling case OS trend** → case reserves being released → prefer incurred-basis or investigate adequacy.
- **Stable case OS** → both paid and incurred are good method inputs.

In [ ]:
# Case outstanding triangle = incurred - paid
case_os = incurred - paid

# Trend summary on case OS
try:
    ts_case = trend_summary(case_os)
    print('=== CASE OUTSTANDING DIAGONAL TREND ===')
    print(ts_case['diagonal_trend'].to_string(float_format=lambda x: f'{x:.6f}'))

    print('\n=== CASE OS: ORIGIN TRENDS (first 8 dev ages) ===')
    print('  Positive = case OS growing across AYs at same dev age (strengthening)')
    print(ts_case['origin_trends'].iloc[:8].to_frame('slope').to_string(float_format=lambda x: f'{x:.6f}'))
except Exception as e:
    print(f'trend_summary on case OS: {e}')

In [ ]:
# Plot case OS levels by accident year at each development age
case_wide = case_os.to_frame(origin_as_datetime=False)
# Show first 8 development ages (first 2 years of quarterly development)
display_ages = list(case_wide.columns[:8])
ay_labels_raw = [str(x)[:4] for x in case_wide.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Case OS by AY at each dev age (each line = one dev age)
ax = axes[0]
cmap = plt.cm.tab10
for i, age in enumerate(display_ages[:6]):
    col_data = case_wide[age].dropna()
    if len(col_data) < 2:
        continue
    ay_nums = [ay_labels_raw.index(str(x)[:4]) for x in col_data.index]
    ax.plot(ay_nums, col_data.values, marker='o', label=f'{age} mo', color=cmap(i), alpha=0.8)
ax.set_xticks(range(len(ay_labels_raw)))
ax.set_xticklabels(ay_labels_raw, rotation=45, fontsize=8)
ax.set_title('Case OS by Accident Year at Fixed Dev Ages')
ax.set_ylabel('Case OS ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=7, title='Dev Age', loc='upper left')
ax.grid(True, alpha=0.3)

# Right: Latest diagonal case OS
latest_case = case_wide.apply(lambda row: row.dropna().iloc[-1] if row.notna().any() else np.nan, axis=1)
ax2 = axes[1]
ax2.bar(range(len(latest_case)), latest_case.values, color='steelblue', alpha=0.85)
ax2.set_xticks(range(len(latest_case)))
ax2.set_xticklabels(ay_labels_raw, rotation=45, fontsize=8)
ax2.set_title('Case OS at Latest Diagonal by AY')
ax2.set_ylabel('Case OS ($)')
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Trend in latest diagonal case OS
valid_case = latest_case.dropna()
if len(valid_case) >= 2:
    slope_case = float(np.polyfit(np.arange(len(valid_case)), valid_case.values, 1)[0])
    print(f'Trend slope of latest-diagonal case OS: {slope_case:+,.0f} per accident year')
    if slope_case > 0:
        print('  Rising case OS → case reserves strengthening. Lean toward PAID-basis methods.')
    else:
        print('  Falling case OS → case reserves weakening. Investigate adequacy.')

### Practice Problem 6: Case Outstanding Trend Analysis

Average case OS at **36 months** of development by accident year (dollar amounts in $000s):

| AY | Avg Case OS at 36 mo |
|---|---|
| 2018 | 8,500 |
| 2019 | 9,200 |
| 2020 | 10,100 |
| 2021 | 11,300 |
| 2022 | 12,500 |

**(a)** Compute the year-over-year percentage growth in average case OS at 36 months.

**(b)** Compute the OLS trend slope of average case OS vs. accident year.

**(c)** If the average claim severity (total losses / claim count) is growing at only 4% per year, what does the 36-month case OS trend imply?

**(d)** Which reserving methods should receive the most weight, and why?

In [ ]:
# Practice Problem 6 — Solution
pp6 = pd.Series([8500, 9200, 10100, 11300, 12500],
                index=[2018, 2019, 2020, 2021, 2022], name='avg_case_os_36mo')

# (a) YoY growth
yoy_case = pp6.pct_change() * 100
print('=== (a) Year-over-Year Growth in Avg Case OS at 36 mo ===')
print(pd.DataFrame({'Avg Case OS': pp6, 'YoY Growth %': yoy_case}).to_string(
    float_format=lambda x: f'{x:,.1f}'))
print(f'  Average YoY growth: {yoy_case.dropna().mean():.1f}%')

# (b) OLS trend slope
x_pp6 = np.arange(len(pp6))
slope_pp6, _ = np.polyfit(x_pp6, pp6.values, 1)
print(f'\n=== (b) OLS Trend Slope ===')
print(f'  Slope = {slope_pp6:+,.0f} per accident year')
print(f'  Implies case OS growing ~{slope_pp6:,.0f}/yr faster than the prior year.')

# (c) Interpretation vs severity trend
severity_growth = 4.0  # % per year
print(f'\n=== (c) Interpretation ===')
avg_yoy = yoy_case.dropna().mean()
excess = avg_yoy - severity_growth
print(f'  Avg case OS growth: {avg_yoy:.1f}%/yr vs. severity trend: {severity_growth:.1f}%/yr')
print(f'  Excess growth (case OS minus severity): {excess:.1f}%/yr')
print('  This excess represents case reserve STRENGTHENING — reserves are being')
print('  set higher than claim severity trends alone would require.')
print('  Effect: incurred triangle LDFs at 36+ months will be biased DOWNWARD over time')
print('  (the reserve build-up itself represents development that won\'t recur).')

# (d) Method preference
print('\n=== (d) Method Selection Implication ===')
print('  Recommended: weight PAID-basis projections most heavily.')
print('  Paid CL and Paid BF are unaffected by case reserve strengthening.')
print('  Incurred CL would understate development because rising case OS in')
print('  older diagonals appears as less future development needed.')
print('  If incurred methods are used, apply B-S reported adjustment first.')

---
## Section 7 — Sensitivity Testing

### Theory

Sensitivity testing quantifies how IBNR changes for reasonable variations in key assumptions. This documents the **parameter uncertainty** component of reserve risk (distinct from process risk from Mack/bootstrap).

Key levers to test:

| Lever | Range to Test | Impact Driver |
|---|---|---|
| `n_periods` (averaging window) | 3, 5, 7, all | Trend in LDFs over accident years |
| Tail factor selection | Constant(1.00), Constant(1.05), Curve(exp), Curve(power) | Mature AYs with significant tail |
| Paid vs incurred basis | CL on paid vs CL on incurred | Case reserve level / adequacy |
| A priori loss ratio (BF/Benktander) | ±10% around selected | Immature AYs with low % paid |

### Exam Documentation Standard
Report:
- Central estimate (selected assumption)
- Low estimate (assumptions that produce lower IBNR)
- High estimate (assumptions that produce higher IBNR)
- Range as % of central estimate
- Qualitative statement of which lever drives the most uncertainty

In [ ]:
# n_periods sensitivity
n_period_results = {}
for n in [3, 5, 7, -1]:
    label = f'n={n}' if n > 0 else 'n=All'
    r, s, _ = run_chain_ladder(paid, development_kwargs={'n_periods': n})
    n_period_results[label] = {'total_ibnr': r.ibnr_total, 'ibnr_by_ay': s['ibnr']}

base_ibnr = n_period_results['n=All']['total_ibnr']
sensitivity_n = pd.DataFrame({
    label: {'Total IBNR': v['total_ibnr'],
            'vs n=All': v['total_ibnr'] / base_ibnr - 1}
    for label, v in n_period_results.items()
}).T

print('=== n_periods SENSITIVITY — Total IBNR ===')
print(sensitivity_n.to_string(float_format=lambda x: f'{x:,.2f}' if abs(x) > 100 else f'{x:.4f}'))

# AY-level IBNR for key n_periods
ibnr_by_ay = pd.DataFrame({
    label: v['ibnr_by_ay']
    for label, v in n_period_results.items()
})
ibnr_by_ay.loc['Total'] = ibnr_by_ay.sum()
print('\n=== IBNR BY ACCIDENT YEAR ===')
print(ibnr_by_ay.to_string(float_format=lambda x: f'{x:,.0f}'))

In [ ]:
# Tail factor sensitivity
tail_results = {}
tail_specs = [
    ('TailConstant(1.00)', 'constant', {'tail': 1.000}),
    ('TailConstant(1.01)', 'constant', {'tail': 1.010}),
    ('TailConstant(1.03)', 'constant', {'tail': 1.030}),
    ('TailCurve(exp)',     'curve',    {'curve': 'exponential'}),
    ('TailCurve(power)',   'curve',    {'curve': 'inverse_power'}),
]
for label, tail_kind, tail_kw in tail_specs:
    try:
        r, _, _ = run_chain_ladder(paid, tail_kind=tail_kind, tail_kwargs=tail_kw)
        tail_results[label] = r.ibnr_total
    except Exception as e:
        print(f'  {label}: {e}')
        tail_results[label] = np.nan

base_notail = n_period_results['n=All']['total_ibnr']  # no-tail baseline
tail_df = pd.DataFrame({'Total IBNR': tail_results}).assign(
    vs_no_tail=lambda d: d['Total IBNR'] / base_notail - 1
)
print('=== TAIL FACTOR SENSITIVITY ===')
print(tail_df.to_string(float_format=lambda x: f'{x:,.2f}' if abs(x) > 1 else f'{x:.4f}'))

In [ ]:
# Tornado chart: combined sensitivity
all_ibnr_values = list(n_period_results[k]['total_ibnr'] for k in n_period_results)
all_ibnr_values += [v for v in tail_results.values() if not np.isnan(v)]

selected_ibnr = base_ibnr  # no-tail, all-period
ibnr_low  = min(all_ibnr_values)
ibnr_high = max(all_ibnr_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: n_periods sensitivity bar
ax = axes[0]
labels_n = list(n_period_results.keys())
values_n = [n_period_results[k]['total_ibnr'] for k in labels_n]
devs_n   = [v - selected_ibnr for v in values_n]
colors_n = ['#d62728' if d > 0 else '#1f77b4' for d in devs_n]
ax.barh(labels_n, devs_n, color=colors_n, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('IBNR deviation from n=All baseline')
ax.set_title('n_periods Sensitivity')
for i, (d, v) in enumerate(zip(devs_n, values_n)):
    ax.text(d * 1.01 if d >= 0 else d * 1.01, i, f'{v:,.0f}', va='center', fontsize=8)
ax.grid(True, axis='x', alpha=0.3)

# Right: tail sensitivity
ax2 = axes[1]
labels_t = [k for k in tail_results if not np.isnan(tail_results[k])]
values_t = [tail_results[k] for k in labels_t]
devs_t   = [v - selected_ibnr for v in values_t]
colors_t = ['#d62728' if d > 0 else '#1f77b4' for d in devs_t]
ax2.barh(labels_t, devs_t, color=colors_t, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('IBNR deviation from no-tail baseline')
ax2.set_title('Tail Factor Sensitivity')
for i, (d, v) in enumerate(zip(devs_t, values_t)):
    ax2.text(d + abs(d) * 0.01, i, f'{v:,.0f}', va='center', fontsize=8)
ax2.grid(True, axis='x', alpha=0.3)

plt.suptitle(f'IBNR Sensitivity Analysis — Selected (baseline): {selected_ibnr:,.0f}', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n=== SENSITIVITY SUMMARY ===')
print(f'  Baseline IBNR (n=All, no tail): {selected_ibnr:>12,.0f}')
print(f'  Low across all combos:          {ibnr_low:>12,.0f}  ({(ibnr_low/selected_ibnr-1)*100:+.1f}%)')
print(f'  High across all combos:         {ibnr_high:>12,.0f}  ({(ibnr_high/selected_ibnr-1)*100:+.1f}%)')
print(f'  Range:                          {ibnr_high-ibnr_low:>12,.0f}  ({(ibnr_high-ibnr_low)/selected_ibnr*100:.1f}% of baseline)')

### Practice Problem 7: Sensitivity Table Interpretation

A chain ladder analysis of paid claims produces the following total IBNR estimates:

| Assumption | Total IBNR |
|---|---|
| n_periods = 3 (last 3 years) | 12,450,000 |
| n_periods = 5 (last 5 years) | 11,820,000 |
| n_periods = All | 11,200,000 |
| + TailCurve(exp) | 11,680,000 |
| + TailConstant(1.05) | 11,760,000 |

**(a)** Compute the range of total IBNR as a percent of the n=All (no-tail) baseline.

**(b)** The n=3 estimate is 11.2% above the n=All estimate. What does this indicate about the LDF trend across accident years?

**(c)** Which parameter (n_periods vs. tail) drives the most uncertainty?

**(d)** State a selected estimate with a documented rationale.

In [ ]:
# Practice Problem 7 — Solution
pp7_results = pd.Series({
    'n=3':                 12_450_000,
    'n=5':                 11_820_000,
    'n=All (baseline)':    11_200_000,
    'n=All + Tail(exp)':   11_680_000,
    'n=All + Tail(1.05)':  11_760_000,
}, name='Total IBNR')

baseline = pp7_results['n=All (baseline)']
pp7_df = pp7_results.to_frame()
pp7_df['vs_baseline_%'] = (pp7_df['Total IBNR'] / baseline - 1) * 100

# (a) Range
ibnr_lo_pp7 = pp7_results.min()
ibnr_hi_pp7 = pp7_results.max()
print('=== (a) IBNR Range ===')
print(pp7_df.to_string(float_format=lambda x: f'{x:,.0f}' if abs(x) > 100 else f'{x:+.1f}%'))
print(f'\n  Range: {ibnr_hi_pp7-ibnr_lo_pp7:,.0f} '
      f'= {(ibnr_hi_pp7-ibnr_lo_pp7)/baseline*100:.1f}% of baseline')

# (b) LDF trend interpretation
print('\n=== (b) n=3 vs n=All Interpretation ===')
pct_diff = (pp7_results['n=3'] / pp7_results['n=All (baseline)'] - 1) * 100
print(f'  n=3 is {pct_diff:+.1f}% above n=All.')
print('  The 3 most recent years have HIGHER LDFs than the long-run average.')
print('  This means link ratios are declining over time (older AYs had lower factors).')
print('  Implication: if the higher recent factors reflect a structural change,')
print('  use n=3 or n=5. If they reflect random noise, use all periods.')

# (c) Parameter drivers
n_spread  = pp7_results['n=3'] - pp7_results['n=All (baseline)']
tl_spread = max(pp7_results['n=All + Tail(exp)'], pp7_results['n=All + Tail(1.05)']) - baseline
print(f'\n=== (c) Parameter Drivers ===')
print(f'  n_periods spread: {n_spread:,.0f}  ({n_spread/baseline*100:.1f}% of baseline)')
print(f'  Tail spread:      {tl_spread:,.0f}  ({tl_spread/baseline*100:.1f}% of baseline)')
print(f'  → {"n_periods" if n_spread > tl_spread else "Tail factor"} drives the most uncertainty.')

# (d) Selected estimate
selected_pp7 = pp7_results['n=5']
print(f'\n=== (d) Selected Estimate ===')
print(f'  Selected: n=5, no tail = {selected_pp7:,.0f}')
print('  Rationale: Recent 5-year factors reflect current settlement patterns')
print('  without over-weighting the very latest 3 years, which may be volatile.')
print('  No tail applied pending additional industry benchmark review.')
print(f'  Range: {ibnr_lo_pp7:,.0f} – {ibnr_hi_pp7:,.0f} '
      f'({(ibnr_hi_pp7-ibnr_lo_pp7)/selected_pp7*100:.1f}% of selected).')

---
## Section 8 — Method Comparison Framework

### Theory

Method selection is driven by **maturity** (% paid at the latest diagonal) and the **diagnostic findings** from Sections 1–7. The structured decision matrix below is the exam-standard approach:

| Maturity | % Paid | Recommended Methods | Notes |
|---|---|---|---|
| Early (12–36 mo) | < 30% | BF, Benktander | CL has low credibility; a priori dominates |
| Developing (36–72 mo) | 30%–70% | CL, BF, Benktander blend | Credibility-weight between CL and BF |
| Mature (72+ mo) | > 70% | CL, Case Outstanding | CL is most credible; BF adds little value |

### Credibility Blending Rule
$$Ultimate_{blended} = w_{CL} \times Ultimate_{CL} + w_{BF} \times Ultimate_{BF}$$
where $w_{CL}$ increases with % paid, and $w_{BF} = 1 - w_{CL}$.

### Exam Response Format
1. State the maturity profile for each accident year.
2. Apply the decision matrix to assign methods.
3. Report CL, BF, and blended ultimates side-by-side.
4. State selected estimate with explicit rationale.
5. Flag any caveats (large losses, data anomalies, trend signals from Sections 1–7).

In [ ]:
# Maturity profile
paid_latest_df     = paid.latest_diagonal.to_frame(origin_as_datetime=False, keepdims=True)
incurred_latest_df = incurred.latest_diagonal.to_frame(origin_as_datetime=False, keepdims=True)

ay_years   = np.array([int(str(x)[:4]) for x in paid_latest_df['origin'].values])
paid_vals  = paid_latest_df['paid'].values.astype(float)
inc_vals   = incurred_latest_df['incurred'].values.astype(float)
pct_paid   = paid_vals / inc_vals

maturity = pd.DataFrame({
    'latest_paid':     paid_vals,
    'latest_incurred': inc_vals,
    'pct_paid':        pct_paid,
    'CL_weight':  np.where(pct_paid > 0.70, 0.80, np.where(pct_paid > 0.40, 0.50, 0.20)),
    'BF_weight':  np.where(pct_paid > 0.70, 0.20, np.where(pct_paid > 0.40, 0.50, 0.80)),
    'method':     np.where(pct_paid > 0.70, 'CL dominant',
                  np.where(pct_paid > 0.40, 'CL/BF blend', 'BF dominant')),
}, index=ay_years)

print('=== MATURITY PROFILE & METHOD WEIGHTS ===')
print(maturity.to_string(float_format=lambda x: f'{x:,.3f}' if isinstance(x, float) else str(x)))

In [ ]:
# Run Chain Ladder and Bornhuetter-Ferguson
apriori_lr = 0.70
exposure_series = pd.Series(paid_vals / apriori_lr, index=ay_years, dtype=float)

# Chain Ladder (all periods, no tail)
r_cl, s_cl, a_cl = run_chain_ladder(paid)

# Bornhuetter-Ferguson
r_bf, s_bf, a_bf = run_bornhuetter_ferguson(
    paid, exposure_series, apriori=apriori_lr
)

# Side-by-side comparison
compare = pd.DataFrame({
    'Latest':       s_cl['latest'],
    'CL_Ultimate':  s_cl['ultimate'],
    'BF_Ultimate':  s_bf['ultimate'],
    'CL_IBNR':      s_cl['ibnr'],
    'BF_IBNR':      s_bf['ibnr'],
})
compare.index = [str(x)[:4] for x in compare.index]
compare['IBNR_Diff_BF_vs_CL'] = compare['BF_IBNR'] - compare['CL_IBNR']
compare.loc['Total'] = compare.sum()

print('=== CL vs BF: ULTIMATE AND IBNR COMPARISON ===')
print(f'  A priori loss ratio: {apriori_lr:.0%}')
print(f'  Positive diff = BF produces more IBNR than CL (BF leans on a priori)')
print(compare.to_string(float_format=lambda x: f'{x:,.0f}'))

In [ ]:
# Credibility-blended ultimate and IBNR
cl_ult = s_cl['ultimate'].values
bf_ult = s_bf['ultimate'].values
w_cl   = maturity['CL_weight'].values
w_bf   = maturity['BF_weight'].values

blended_ult  = w_cl * cl_ult + w_bf * bf_ult
blended_ibnr = blended_ult - paid_vals

blend_df = pd.DataFrame({
    'pct_paid':     pct_paid,
    'w_CL':         w_cl,
    'CL_Ult':       cl_ult,
    'BF_Ult':       bf_ult,
    'Blended_Ult':  blended_ult,
    'Blended_IBNR': blended_ibnr,
}, index=[str(x)[:4] for x in ay_years])
blend_df.loc['Total'] = blend_df[['CL_Ult', 'BF_Ult', 'Blended_Ult', 'Blended_IBNR']].sum()
blend_df.at['Total', 'pct_paid'] = np.nan
blend_df.at['Total', 'w_CL'] = np.nan

print('=== CREDIBILITY-BLENDED RESULTS ===')
print(blend_df.to_string(float_format=lambda x: f'{x:,.2f}' if pd.notna(x) and abs(x) < 10 else f'{x:,.0f}'))

# Summary
total_cl_ibnr     = r_cl.ibnr_total
total_bf_ibnr     = r_bf.ibnr_total
total_blend_ibnr  = blended_ibnr.sum()
print(f'\n  CL Total IBNR:      {total_cl_ibnr:>12,.0f}')
print(f'  BF Total IBNR:      {total_bf_ibnr:>12,.0f}')
print(f'  Blended Total IBNR: {total_blend_ibnr:>12,.0f}')

In [ ]:
# Snapshot and reconciliation: compare 5-year CL (current) vs all-year CL (baseline)
r_cl5, s_cl5, a_cl5 = run_chain_ladder(paid, development_kwargs={'n_periods': 5})
r_all, s_all, a_all = run_chain_ladder(paid)

snap_5yr = snapshot_method_output(r_cl5, s_cl5, a_cl5)
snap_all = snapshot_method_output(r_all, s_all, a_all)

# build_reconciliation_report compares current (5yr) vs baseline (all-year)
# Both must share the same method_name key
recon = build_reconciliation_report(
    {'Chain Ladder (Track A)': snap_5yr},
    {'Chain Ladder (Track A)': snap_all},
)

failed = recon[~recon['passed']]
print(f'Reconciliation: {len(recon)} checks, {len(failed)} failed '
      f'(tolerance: absolute pattern diff and relative total/AY diff)')
print(f'\nFailed checks (5-year window vs all-year baseline):')
if len(failed) > 0:
    show_cols = ['metric_name', 'metric_group', 'baseline_value', 'current_value', 'rel_diff', 'reason_code']
    display_cols = [c for c in show_cols if c in failed.columns]
    print(failed[display_cols].head(12).to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))
else:
    print('  No failed checks — 5-year and all-year windows produce equivalent results.')

### Practice Problem 8: Credibility-Blended Reserve Recommendation

Given ultimates from three methods for five accident years:

| AY | Latest Reported | % Paid | CL Ultimate | BF Ultimate |
|---|---|---|---|---|
| 2019 | 4,900,000 | 88% | 5,200,000 | 5,180,000 |
| 2020 | 4,600,000 | 76% | 4,800,000 | 4,750,000 |
| 2021 | 3,800,000 | 62% | 4,500,000 | 4,400,000 |
| 2022 | 3,000,000 | 45% | 4,200,000 | 4,000,000 |
| 2023 | 1,500,000 | 28% | 4,100,000 | 3,700,000 |

Use the maturity-based weight scheme:
- % Paid > 70%: CL weight = 0.80, BF weight = 0.20
- % Paid 40–70%: CL weight = 0.50, BF weight = 0.50
- % Paid < 40%: CL weight = 0.20, BF weight = 0.80

**(a)** Assign weights to each accident year.

**(b)** Compute the credibility-blended ultimate for each AY.

**(c)** Compute blended IBNR = blended ultimate − latest reported.

**(d)** State a reserve recommendation: blended total IBNR plus one qualitative caveat.

In [ ]:
# Practice Problem 8 — Solution
pp8 = pd.DataFrame({
    'latest':      [4_900_000, 4_600_000, 3_800_000, 3_000_000, 1_500_000],
    'pct_paid':    [0.88, 0.76, 0.62, 0.45, 0.28],
    'CL_ultimate': [5_200_000, 4_800_000, 4_500_000, 4_200_000, 4_100_000],
    'BF_ultimate': [5_180_000, 4_750_000, 4_400_000, 4_000_000, 3_700_000],
}, index=[2019, 2020, 2021, 2022, 2023])

# (a) Assign weights
pp8['w_CL'] = np.where(pp8['pct_paid'] > 0.70, 0.80,
              np.where(pp8['pct_paid'] > 0.40, 0.50, 0.20))
pp8['w_BF'] = 1.0 - pp8['w_CL']

print('=== (a) Weight Assignment ===')
print(pp8[['pct_paid', 'w_CL', 'w_BF']].to_string(float_format=lambda x: f'{x:.2f}'))

# (b) Blended ultimate
pp8['blended_ult'] = pp8['w_CL'] * pp8['CL_ultimate'] + pp8['w_BF'] * pp8['BF_ultimate']

# (c) Blended IBNR
pp8['blended_IBNR'] = pp8['blended_ult'] - pp8['latest']
pp8['CL_IBNR']      = pp8['CL_ultimate'] - pp8['latest']
pp8['BF_IBNR']      = pp8['BF_ultimate'] - pp8['latest']

totals = pp8[['latest', 'CL_ultimate', 'BF_ultimate', 'blended_ult',
               'CL_IBNR', 'BF_IBNR', 'blended_IBNR']].sum().rename('Total')
result = pd.concat([pp8[['latest', 'CL_ultimate', 'BF_ultimate', 'blended_ult',
                           'CL_IBNR', 'BF_IBNR', 'blended_IBNR']], totals.to_frame().T])

print('\n=== (b)/(c) Blended Ultimate and IBNR ===')
print(result.to_string(float_format=lambda x: f'{x:,.0f}'))

# (d) Recommendation
total_blend_ibnr_pp8 = pp8['blended_IBNR'].sum()
total_cl_pp8         = pp8['CL_IBNR'].sum()
total_bf_pp8         = pp8['BF_IBNR'].sum()
print(f'\n=== (d) Reserve Recommendation ===')
print(f'  Selected Total IBNR: {total_blend_ibnr_pp8:,.0f} (credibility-blended)')
print(f'  CL indication:       {total_cl_pp8:,.0f}')
print(f'  BF indication:       {total_bf_pp8:,.0f}')
print(f'  Blend is {(total_blend_ibnr_pp8/total_cl_pp8-1)*100:+.1f}% vs CL and '
      f'{(total_blend_ibnr_pp8/total_bf_pp8-1)*100:+.1f}% vs BF.')
print('\n  Caveat: AY 2023 (28% paid) is highly sensitive to the a priori loss ratio.')
print('  The BF weight of 80% for that year means the reserve is driven by the')
print('  assumed ECR rather than the emerging triangle data. Revisit after')
print('  additional quarters of development are available.')